In [ ]:
# Importing all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# reading the dataset

csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
# inspecting the first few rows

df.head()

In [ ]:
# Task 3: Write your code here:
# viewing the dataset information

df.info()

In [ ]:
# Task 4: Write your code here:
# Show statistical description using describe()

df.describe()

In [ ]:
# Task 5: Write your code here:
# Plot the target distribution (delivery_time)

def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
# drop the order ID column from the data

df = df.drop(columns = "Order_ID", axis = 1)
df.head()

In [ ]:
# Task 2: Write your code here:
# check for missing values in the dataset
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

# handle if any
# drop the samples with missing delivery time
df = df.dropna(subset = "Delivery_Time")

# check again
check_missing_values(df)

# Fill categorical columns with 'unknown' - missing likely means "not specified"
for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    df[col] = df[col].fillna('unknown')

# Fill courier experience years with mode - discrete feature, mode is most representative
df['Courier_Experience_yrs'] = df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].mode()[0])

check_missing_values(df)


In [ ]:
# Task 3: Write your code here:
# check for duplicates and remove them
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:
# encode catrgorical variables

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder

# check and print the categorical columns
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

# use one hot encoding
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col].astype(str))

  # onehot_encoder = OneHotEncoder(sparse_output=False)
  # df[col] = onehot_encoder.fit_transform(df[col].astype(str))


df.head()

In [ ]:
# Task 5: Write your code here:
# Apply feature scaling for all features

from sklearn.preprocessing import StandardScaler #import StandardScaler

scaler = StandardScaler() # Instantiate StandardScaler
features = df.columns.drop("Delivery_Time") # remove the target
df[features] = scaler.fit_transform(df[features])
df.head()

In [ ]:
# Task 6: Write your code here:
# Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)
# not needed

In [ ]:
# Task 1: Write your code here:
# splittind dataset into X and y

X = df.drop("Delivery_Time", axis=1).astype(float)
y = df["Delivery_Time"].astype(float)

print(X.shape)
y.shape

In [ ]:
# importing the needed libraries
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor

# Task 2,3,4,5: Write your code here:

# splitting data by KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# train a RandomForest Model
model = RandomForestRegressor(n_estimators=200)

mae_list = []

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/5")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  model.fit(X_train, y_train)

  # Predict
  y_pred = model.predict(X_test)

  # Calculate metrics
  mae = mean_absolute_error(y_test, y_pred)

  print(mae)

  # Store results
  mae_list.append(mae)


# print the averages score across all folds

In [ ]:
# Gather importances from the models (from the last fold)
importances = {}

importances = model.feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
# Task 5: Write your code here:
# Plot the target distribution (delivery_time)

def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

y_pred = model.predict(X_test)

check_target_distribution(df, y_pred)


In [ ]:
# Task Bonus: Write your code here: